# 03 – GIS Visualisation

Create interactive Folium maps with markers, heatmaps, and export to
GeoJSON / Shapefile.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import numpy as np
import geopandas as gpd
from shapely.geometry import Point

from src.visualization.maps import (
    create_base_map, add_soundscape_markers, add_heatmap, save_map
)
from src.visualization.gis_export import export_geojson, export_shapefile
from src.utils.logger import setup_logger

setup_logger(level='INFO')
print('Imports OK')

## 1. Synthetic Data

In [ ]:
rng = np.random.default_rng(7)
n = 25
lons = rng.uniform(72.0, 73.5, n)
lats = rng.uniform(34.5, 35.5, n)
aci = rng.uniform(200, 800, n)
cluster = rng.integers(0, 4, n)

gdf = gpd.GeoDataFrame(
    {'aci': aci, 'cluster': cluster,
     'site': [f'Site_{i:02d}' for i in range(n)]},
    geometry=[Point(lon, lat) for lon, lat in zip(lons, lats)],
    crs='EPSG:4326'
)
print(f'GeoDataFrame: {len(gdf)} rows')

## 2. Interactive Folium Map

In [ ]:
m = create_base_map(center=[35.0, 72.5], zoom_start=9)
m = add_soundscape_markers(m, gdf, label_col='site',
                            color_col='cluster', popup_cols=['aci', 'cluster'])
m = add_heatmap(m, gdf, value_col='aci')

import folium
folium.LayerControl().add_to(m)
m   # renders in Jupyter

## 3. Save Map to HTML

In [ ]:
out = pathlib.Path('../outputs/notebooks_map.html')
save_map(m, out)
print(f'Saved: {out}')

## 4. GIS Exports

In [ ]:
import pathlib
out_dir = pathlib.Path('../outputs')

geojson_path = export_geojson(gdf, out_dir / 'demo_soundscape.geojson',
                               metadata={'notebook': '03_visualization'})
print(f'GeoJSON: {geojson_path}')

shp_path = export_shapefile(gdf, out_dir / 'demo_soundscape.shp')
print(f'Shapefile: {shp_path}')